# Path A, P7e — per-file interface table (Oct 26, 2026)

Runs: `load_runs("behavior", roles={"train", "val"}, keep_bot=True, keep_incomplete=True)`: training topologies (Hierarchy, Full, Star), train + val groups. Raw rows (no lab clock), human seats only.

- **Move:** a row where a human player's s changes (|Δs| > 0). **Drag move:** a move less than 0.3 s after the same player's previous move in the round (moves at the same time as the previous one are left out).
- **Step size:** mode of |Δs| over all moves (3 decimals). **Modal pace:** mode over drag moves of (|Δs| / step size) / gap, steps/s (1 decimal). **Speed** = step size x modal pace (units/s). Ties go to the smaller value.
- **% rows with > 1 mover:** rows (from the second of each round) where more than one human player moves. **% on grid:** human values with |10 s − round(10 s)| <= 1e-6.
- **Constancy:** each round's step size and modal pace (its own moves). A file is flagged if some round's step size differs from the file's or its modal pace differs from the file's by more than 30%. Rounds with no move or no drag move are listed as unchecked.

Function: `interface.interface_table` (`src/slider/interface.py`). Saved: `results/tables/interface_table.csv`.

In [1]:
import numpy as np
import pandas as pd

from slider import config
from slider import interface
from slider import slider_logs as sl

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

runs, table = sl.load_runs("behavior", roles={"train", "val"}, keep_bot=True, keep_incomplete=True)
it = interface.interface_table(runs)
it.drop(columns=["group_id"]).round(3)

,file,dataset,split,version,rounds,rows,time_decimals,median_gap_s,pct_rows_multi_mover,step_size,pct_on_grid,modal_pace,speed,flagged,differing_rounds,unchecked_rounds
0,log_v09_Fri Nov 04 155722 EDT 2016.txt,EL,train,v09,3,615,2,0.100,42.810,0.10,100.000,10.0,1.00,True,wave 9: pace 20,
1,log_v09_Fri Nov 04 165730 EDT 2016.txt,EL,train,v09,3,364,2,0.100,24.654,0.10,100.000,10.0,1.00,True,wave 2: pace 5,
2,log_v09_Fri Nov 11 155640 EST 2016.txt,EL,val,v09,3,1979,2,0.100,21.660,0.10,100.000,10.0,1.00,True,wave 4: pace 20,
3,log_v09_Fri Nov 11 160012 EST 2016.txt,EL,train,v09,3,307,2,0.050,25.000,0.10,100.000,20.0,2.00,False,,
4,log_v09_Fri Nov 11 165515 EST 2016.txt,EL,train,v09,3,419,2,0.100,33.894,0.10,100.000,10.0,1.00,True,wave 9: pace 20,
5,log_v09_Mon Nov 07 164258 EST 2016.txt,EL,train,v09,3,719,2,0.100,21.369,0.10,100.000,10.0,1.00,True,wave 4: pace 20,
6,log_v09_Mon Nov 07 175505 EST 2016.txt,EL,train,v09,3,1992,2,0.100,30.065,0.10,100.000,5.0,0.50,True,wave 4: pace 10,
7,log_v09_Mon Oct 31 164716 EDT 2016.txt,EL,val,v09,3,1797,2,0.100,16.332,0.10,100.000,10.0,1.00,False,,
8,log_v09_Wed Nov 02 170112 EDT 2016.txt,EL,val,v09,3,1250,2,0.100,37.129,0.10,100.000,10.0,1.00,True,wave 3: pace 20,
9,log_v09_Wed Nov 02 173358 EDT 2016.txt,EL,train,v09,3,1509,2,0.100,25.631,0.10,100.000,10.0,1.00,True,wave 2: pace 20,


## Summary by dataset, version tag and step size

In [2]:
def counts(x):
    """Distinct values and how many files have each, e.g. "10 (14), 20 (1)"."""
    v = x.round(3).value_counts()
    return ", ".join(f"{value:g} ({n})" for value, n in v.items())


it.groupby(["dataset", "version", "step_size"], sort=False).agg(
    files=("file", "size"), rounds=("rounds", "sum"), flagged=("flagged", "sum"),
    modal_pace=("modal_pace", counts), speed=("speed", counts))

files  rounds  flagged                               modal_pace                                    speed
dataset version step_size                                                                                                          
EL      v09     0.10          17      51       11                   10 (15), 20 (1), 5 (1)                   1 (15), 2 (1), 0.5 (1)
CL      none    0.02           5      29        0                                   10 (5)                                  0.2 (5)
        v09     0.10          13      78        8                                  10 (13)                                   1 (13)
CC      none    0.10          25     216        5  30.3 (19), 29.4 (3), 33.3 (2), 32.3 (1)  3.03 (19), 2.94 (3), 3.33 (2), 3.23 (1)

## CL files without a version tag

In [3]:
it[(it["dataset"] == "CL") & (it["version"] == "none")].drop(columns=["group_id"]).round(3)

,file,dataset,split,version,rounds,rows,time_decimals,median_gap_s,pct_rows_multi_mover,step_size,pct_on_grid,modal_pace,speed,flagged,differing_rounds,unchecked_rounds
17,log_Fri Sep 16 165753 EDT 2016.txt,CL,val,none,6,4465,2,0.1,63.288,0.02,60.602,10.0,0.2,False,,
18,log_Mon Sep 19 164446 EDT 2016.txt,CL,val,none,5,7045,2,0.1,47.060,0.02,61.460,10.0,0.2,False,,
19,log_Mon Sep 19 173635 EDT 2016.txt,CL,val,none,6,2790,2,0.1,60.489,0.02,62.079,10.0,0.2,False,,
20,log_Tue Sep 20 142130 EDT 2016.txt,CL,train,none,6,3597,2,0.1,48.399,0.02,65.662,10.0,0.2,False,,
21,log_Wed Sep 21 160602 EDT 2016.txt,CL,train,none,6,6463,2,0.1,62.614,0.02,50.585,10.0,0.2,False,,


## Flagged files

The rounds that differ, then every round of each flagged file: rows, median row gap, step size, modal pace, drag moves.

In [4]:
flagged = it[it["flagged"]]
flagged[["file", "dataset", "version", "step_size", "modal_pace", "differing_rounds", "unchecked_rounds"]]

,file,dataset,version,step_size,modal_pace,differing_rounds,unchecked_rounds
0,log_v09_Fri Nov 04 155722 EDT 2016.txt,EL,v09,0.1,10.0,wave 9: pace 20,
1,log_v09_Fri Nov 04 165730 EDT 2016.txt,EL,v09,0.1,10.0,wave 2: pace 5,
2,log_v09_Fri Nov 11 155640 EST 2016.txt,EL,v09,0.1,10.0,wave 4: pace 20,
4,log_v09_Fri Nov 11 165515 EST 2016.txt,EL,v09,0.1,10.0,wave 9: pace 20,
5,log_v09_Mon Nov 07 164258 EST 2016.txt,EL,v09,0.1,10.0,wave 4: pace 20,
6,log_v09_Mon Nov 07 175505 EST 2016.txt,EL,v09,0.1,5.0,wave 4: pace 10,
8,log_v09_Wed Nov 02 170112 EDT 2016.txt,EL,v09,0.1,10.0,wave 3: pace 20,
9,log_v09_Wed Nov 02 173358 EDT 2016.txt,EL,v09,0.1,10.0,wave 2: pace 20,
10,log_v09_Wed Nov 09 154710 EST 2016.txt,EL,v09,0.1,10.0,wave 3: pace 20,
13,log_v09_Wed Nov 16 154247 EST 2016.txt,EL,v09,0.1,10.0,wave 2: pace 20,


In [5]:
rows = []
for run in runs:
    if run["file"] in set(flagged["file"]):
        m = interface.moves(run)
        step, pace = interface.step_and_pace(m)
        rows.append({"file": run["file"], "wave": run["wave"], "rows": len(run["t"]),
                     "median_gap_s": np.median(np.diff(run["t"])), "step_size": step, "modal_pace": pace,
                     "drag_moves": int(np.sum((m["gap"] > 0) & (m["gap"] < interface.DRAG))),
                     "bot_seats": ", ".join(str(b) for b in sorted(run["bot_players"]))})
pd.DataFrame(rows).set_index(["file", "wave"]).round(3)

rows  median_gap_s  step_size  modal_pace  drag_moves bot_seats
file                                   wave                                                                 
log_v09_Fri Nov 04 155722 EDT 2016.txt 2      445         0.100        0.1        10.0         541          
                                       6      123         0.100        0.1        10.0         120          
                                       9       47         0.100        0.1        20.0          51          
log_v09_Fri Nov 04 165730 EDT 2016.txt 2      241         0.110        0.1         5.0         156          
                                       6       69         0.100        0.1        10.0          82          
                                       9       54         0.100        0.1        10.0          69          
log_v09_Fri Nov 11 155640 EST 2016.txt 2      810         0.100        0.1        10.0         605          
                                       4      161         0.100        0.1        20.0         124          
                                       8     1008         0.190        0.1        10.0         692          
log_v09_Fri Nov 11 165515 EST 2016.txt 2      228         0.100        0.1        10.0         243          
                                       6      107         0.100        0.1        10.0         117          
                                       9       84         0.060        0.1        20.0          79          
log_v09_Fri Oct 07 162937 EDT 2016.txt 2      325         0.100        0.1        10.0         278          
                                       3      428         0.100        0.1        10.0         415          
                                       4      101         0.080        0.1        20.0          93          
                                       5       62         0.100        0.1        10.0          60          
                                       7     1272         0.100        0.1        10.0        1151          
                                       9      126         0.100        0.1        10.0         141          
log_v09_Fri Sep 23 155758 EDT 2016.txt 2      105         0.050        0.1        20.0          94          
                                       3      777         0.100        0.1        10.0         679          
                                       4      114         0.100        0.1        10.0         102          
                                       5     1278         0.100        0.1        10.0        1340          
                                       7      177         0.100        0.1        10.0         178          
                                       9      162         0.100        0.1        10.0         157          
log_v09_Fri Sep 23 170644 EDT 2016.txt 1      748         0.100        0.1        10.0         800          
                                       2       88         0.050        0.1        20.0          86          
                                       3      127         0.100        0.1        20.0         144          
                                       4       43         0.050        0.1        20.0          52          
                                       6       85         0.100        0.1        10.0         104          
                                       8      111         0.100        0.1        10.0         172          
log_v09_Mon Nov 07 164258 EST 2016.txt 2      430         0.200        0.1        10.0         231          
                                       4       45         0.100        0.1        20.0          43          
                                       8      244         0.100        0.1        10.0         285          
log_v09_Mon Nov 07 175505 EST 2016.txt 2     1587         0.100        0.1         5.0        1591          
                                       4      242         0.100        0.1        10.0         228          
            

## EL / CL / CC files with no rows in this load

All log files that `slider_logs.dataset_label` puts in EL, CL or CC, minus the files above. For each: its split G label and the topologies, rows and timestamps of its rounds (the facts `load_runs` filters on).

In [6]:
lab, cloud = sl.log_files()
split = dict(pd.read_csv(config.SPLIT_G)[["group_id", "split"]].to_numpy())
missing = []
for path in lab + cloud:
    dataset = sl.dataset_label(path)
    if dataset in interface.DATASETS and path.name not in set(it["file"]):
        rounds = sl.parse_lab(path) if path in lab else sl.parse_cloud(path)
        t = np.concatenate([r["t"] for r in rounds]) if rounds else np.array([])
        missing.append({"file": path.name, "dataset": dataset,
                        "split G": split.get(path.name, "not in split G"),
                        "rounds": len(rounds), "rows": len(t),
                        "topologies": ", ".join(sorted({sl.label_topology(r["A"]) for r in rounds})),
                        "all times whole seconds": bool(len(t) > 0 and np.all(t == np.round(t)))})
print(len(it), "files in this load;", len(missing), "EL/CL/CC files with no rows in it")
pd.DataFrame(missing)

60 files in this load; 3 EL/CL/CC files with no rows in it


,file,dataset,split G,rounds,rows,topologies,all times whole seconds
0,log_Fri Sep 16 161000 EDT 2016.txt,CL,train,9,7458,"Full, Hierarchy, Ring",True
1,log_Wed Sep 14 155221 EDT 2016.txt,CL,train,9,5036,"Full, Hierarchy, Ring",True
2,log_Wed Sep 14 170705 EDT 2016.txt,CL,train,7,6479,"Full, Hierarchy, Ring",True


## Save

In [7]:
it.to_csv(config.INTERFACE_TABLE, index=False)
print(config.INTERFACE_TABLE.name, len(it), "rows")

interface_table.csv 60 rows
